# B2B BNPL Credit Model — Training Notebook
Run this notebook **once** to train and save the model.
The saved files will be used by the Streamlit app.

In [1]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report, mean_absolute_error, r2_score
import xgboost as xgb

print('Libraries loaded ✅')

Libraries loaded ✅


## Step 1 — Load Dataset

In [2]:
df = pd.read_csv('b2b_bnpl_dataset.csv')
print(f'Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head(3)

Dataset loaded: 10,000 rows × 42 columns


,industry,business_age_years,num_employees,business_type,gst_registered,region,annual_revenue_lakh,gross_margin_pct,operating_cash_flow_lakh,revenue_growth_yoy_pct,...,repeat_order_rate_pct,platform_gmv_lakh,last_order_days_ago,support_tickets_open,sector_bnpl_penetration_pct,macro_gdp_growth_pct,default_probability,bnpl_approved,credit_limit_lakh,credit_source
0,Retail (B2B),2.8,5,Pvt Ltd,1,Rural,966.68,27.04,242.06,-30.07,...,71.0,797.12,34,0,28,4.84,0.4695,0,0.00,none
1,Professional Services,2.5,23,Pvt Ltd,1,Tier-1,144.52,28.20,17.22,13.31,...,63.6,3.68,4,0,30,7.20,0.3011,1,14.61,platform
2,Construction,1.5,51,Sole Prop,1,Metro,53.29,13.84,3.48,30.63,...,63.1,138.92,0,0,10,5.43,0.3309,1,9.52,platform


## Step 2 — Select Features
Only keeping simple, business-readable features (no industry/region/type).

In [3]:
FEATURES = [
    'business_age_years',        # How old is the business
    'annual_revenue_lakh',       # Yearly revenue
    'num_employees',             # Team size
    'gross_margin_pct',          # Profit margin
    'debt_to_equity_ratio',      # Debt load
    'current_ratio',             # Liquidity
    'gst_registered',            # GST compliance
    'pct_invoices_paid_on_time', # Payment discipline
    'avg_payment_delay_days',    # How late payments are
    'months_on_platform',        # Platform experience
    'credit_bureau_score',       # Credit score
    'prev_bnpl_defaults',        # Past defaults
    'num_credit_enquiries_6m',   # Recent credit checks
    'existing_loan_lakh',        # Current loan burden
]

X  = df[FEATURES]
y  = df['bnpl_approved']          # Classification target
yr = df['default_probability']    # Regression target

print('Features selected:', FEATURES)
print('\nClass distribution:')
print(y.value_counts())

Features selected: ['business_age_years', 'annual_revenue_lakh', 'num_employees', 'gross_margin_pct', 'debt_to_equity_ratio', 'current_ratio', 'gst_registered', 'pct_invoices_paid_on_time', 'avg_payment_delay_days', 'months_on_platform', 'credit_bureau_score', 'prev_bnpl_defaults', 'num_credit_enquiries_6m', 'existing_loan_lakh']

Class distribution:
bnpl_approved
1    6050
0    3950
Name: count, dtype: int64


## Step 3 — Train / Test Split

In [4]:
X_train, X_test, y_train, y_test, yr_train, yr_test = train_test_split(
    X, y, yr, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size : {len(X_train):,}')
print(f'Test  size : {len(X_test):,}')

Train size : 8,000
Test  size : 2,000


## Step 4 — Train Approval Classifier (XGBoost)

In [5]:
clf = xgb.XGBClassifier(
    n_estimators    = 300,
    max_depth       = 6,
    learning_rate   = 0.05,
    subsample       = 0.8,
    colsample_bytree= 0.8,
    eval_metric     = 'auc',
    random_state    = 42,
    verbosity       = 0
)

clf.fit(X_train, y_train)

y_pred  = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]
auc     = roc_auc_score(y_test, y_proba)

print(f'✅ Classifier AUC-ROC : {auc:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['Rejected','Approved']))

✅ Classifier AUC-ROC : 0.9827

              precision    recall  f1-score   support

    Rejected       0.93      0.89      0.91       790
    Approved       0.93      0.95      0.94      1210

    accuracy                           0.93      2000
   macro avg       0.93      0.92      0.93      2000
weighted avg       0.93      0.93      0.93      2000



## Step 5 — Train Default Risk Regressor (XGBoost)

In [6]:
reg = xgb.XGBRegressor(
    n_estimators    = 300,
    max_depth       = 6,
    learning_rate   = 0.05,
    subsample       = 0.8,
    colsample_bytree= 0.8,
    random_state    = 42,
    verbosity       = 0
)

reg.fit(X_train, yr_train)

yr_pred = reg.predict(X_test)
mae     = mean_absolute_error(yr_test, yr_pred)
r2      = r2_score(yr_test, yr_pred)

print(f'✅ Regressor MAE : {mae:.4f}')
print(f'✅ Regressor R²  : {r2:.4f}')

✅ Regressor MAE : 0.0374
✅ Regressor R²  : 0.9552


## Step 6 — Save Models & Feature List

In [7]:
joblib.dump(clf,      'bnpl_classifier.pkl')
joblib.dump(reg,      'bnpl_risk_regressor.pkl')
joblib.dump(FEATURES, 'bnpl_features.pkl')

print('Model files saved:')
print('  bnpl_classifier.pkl      — approval classifier')
print('  bnpl_risk_regressor.pkl  — default risk regressor')
print('  bnpl_features.pkl        — feature list')
print()
print('Now run:  streamlit run b2b_bnpl_app.py')

Model files saved:
  bnpl_classifier.pkl      — approval classifier
  bnpl_risk_regressor.pkl  — default risk regressor
  bnpl_features.pkl        — feature list

Now run:  streamlit run b2b_bnpl_app.py


## Step 7 — Quick Sanity Check

In [8]:
# Load back and test
loaded_clf  = joblib.load('bnpl_classifier.pkl')
loaded_reg  = joblib.load('bnpl_risk_regressor.pkl')
loaded_feat = joblib.load('bnpl_features.pkl')

sample = X_test.iloc[:1]
print('Sample prediction:')
print(f'  Approved     : {loaded_clf.predict(sample)[0]}')
print(f'  Approval Prob: {loaded_clf.predict_proba(sample)[0,1]:.2%}')
print(f'  Default Risk : {loaded_reg.predict(sample)[0]:.2%}')
print()
print('✅ Models loaded and working correctly!')

Sample prediction:
  Approved     : 1
  Approval Prob: 88.88%
  Default Risk : 32.63%

✅ Models loaded and working correctly!
